In [ ]:
import subprocess
import json
import os
from hipporag import HippoRAG


 # Decrypt and load environment variables from dotenvx
result = subprocess.run(['dotenvx', 'get', '--format', 'json'], capture_output=True,
text=True)

if result.returncode == 0:
    env_vars = json.loads(result.stdout)
    for key, value in env_vars.items():
        os.environ[key] = value
    print("✅ dotenvx variables loaded successfully!")
else:
    print(f"❌ Error loading dotenvx variables: {result.stderr}")

In [ ]:
%load_ext autoreload
%autoreload 2

import asyncio
import datetime
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import openai
from tenacity import retry, wait_exponential, stop_after_attempt, retry_if_exception_type
from src.config import Config
from src.clients import get_clients
from src.utils import process_pptx_file, generate_prompt
from src.rag_adapters import VectorRAGAdapter, LightRAGAdapter, HippoRAGAdapter, SpannerGraphRAGAdapter, AgenticRAGAdapter
from src.evaluation import add_eval_dataset, evaluate_retrieval
from vectorGraphRag import initialize_lightrag, initialize_dataframe

# Initialize graphRAG and DataFrame

In [ ]:
Config.setup_directories()
clients = get_clients()

hipporag = HippoRAG(
    # llm_model_name="gpt-4o-mini-2024-07-18",
    # embedding_model_name="text-embedding-3-small"
    llm_model_name=Config.LLM_MODEL,
    llm_base_url=Config.LLM_BINDING_HOST,
    embedding_model_name=Config.EMBEDDING_MODEL,
    embedding_base_url=Config.EMBEDDING_BINDING_HOST
)

lightrag = await initialize_lightrag()

adapters = {
    "vector_rag": VectorRAGAdapter(clients.qdrant_client, clients.embedding_service),
    "lightrag": LightRAGAdapter(lightrag),
    "hipporag": HippoRAGAdapter(hipporag),
    "spanner_graph": SpannerGraphRAGAdapter(
        clients.graph_store,
        clients.embedding_service,
        clients.llm_transformer,
        Config.GRAPH_NAME
    ),
    "agentic_rag": AgenticRAGAdapter(clients.qdrant_client, clients.embedding_service)
}

df = initialize_dataframe("./templates/CrossDoc_RAG_Benchmark.csv", list(adapters.keys()))

directory_path = "./documents_to_index"
file_paths = [
    os.path.join(directory_path, f)
    for f in os.listdir(directory_path)
    if os.path.isfile(os.path.join(directory_path, f))
]

contents = process_pptx_file(file_paths)



# Generate vector and graph RAG indexes


In [ ]:
indexing_intervals = {}

for name, adapter in adapters.items():
    start_time = datetime.datetime.now()
    await adapter.index(contents)
    indexing_intervals[name] = (datetime.datetime.now() - start_time).total_seconds()

index_interval_dataframe = pd.DataFrame(
    indexing_intervals,
    index=["indexing_time"]
)


# Retrieve contexts

In [ ]:
retrieval_intervals = {name: [] for name in adapters.keys()}
retrieval_costs = {name: [] for name in adapters.keys()}

for i in range(df.shape[0]):
    query = str(df.at[i, "query"])
    await asyncio.sleep(1)

    for name, adapter in adapters.items():
        start_time = datetime.datetime.now()
        cost, context = await adapter.retrieve(query)
        retrieval_intervals[name].append((datetime.datetime.now() - start_time).total_seconds()) 
        retrieval_costs[name].append(cost)
        df.at[i, f"results_{name}"] = context
        


retrieval_interval_dataframe = pd.DataFrame(
    retrieval_intervals,
    index=list(df["query_id"])
)

    

# Print retrieval results

In [ ]:
for i in range(df.shape[0]):
    print(f"\nQuery: {df.at[i, "query"]}\n")
    for name in adapters.keys():
        print(f"results from {name}: {df.at[i, f'results_{name}']}\n\n")

# Generate LLM responses

In [ ]:
async_openai_client = clients.async_openai_client

@retry(wait=wait_exponential(min=4, max=60), stop=stop_after_attempt(10), retry=retry_if_exception_type(openai.RateLimitError))
async def safe_completion(prompt: str):
    return await async_openai_client.responses.create(
        model=Config.LLM_MODEL, # type:ignore
        input=[{"role":"user", "content":prompt}]
    )
for i in range(df.shape[0]):
    query = str(df.at[i, "query"])

    tasks = []
    names = []
    await asyncio.sleep(1)
    for name in adapters.keys():
        if name == "agentic_rag":
            df.at[i, f"actual_responses_{name}"] = adapters[name].get_response(query)
            continue
        context = df.at[i, f"results_{name}"]
        prompt = generate_prompt(query, context) # type: ignore
        names.append(name)
        tasks.append(safe_completion(prompt))
    
    responses = await asyncio.gather(*tasks)
    for name, response in zip(names, responses):
        df.at[i, f"actual_responses_{name}"] = response.output_text
        

# Add dataset objects to DataFrame for eval

In [ ]:
strategies = list(adapters.keys())
add_eval_dataset(df, strategies)

# Perform retrieval evaluation

In [ ]:
retrieval_interval_dataframe = await evaluate_retrieval(df, retrieval_interval_dataframe, async_openai_client, strategies)

# Create interval DataFrame 

In [ ]:

interval_data = pd.concat([index_interval_dataframe.T, retrieval_interval_dataframe.T], axis=1)

# Generate interval plots

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12,5))
plot1 = sns.barplot(x=interval_data.index, y=interval_data["retrieval_time"], ax=axes[0])
axes[0].set_title("Retrieval Interval")
plot2 = sns.barplot(x=interval_data.index, y=interval_data["indexing_time"], ax=axes[1])
axes[1].set_title("Indexing Interval")

plot1.set(xlabel="RAG solutions", ylabel="Seconds")
plot2.set(xlabel="RAG solutions", ylabel="Seconds")
plt.tight_layout()
plt.show()